In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm
import statsmodels.api as sm

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config

c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [4]:
# set seed
random.seed(42)

# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

# select a response variable (either marekt return or sp500 return)
y = response_variables['vwretx']  # or 'sprtrn' for SP500 returns or vwretx

# # transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# Final filtered dataframe
X = X[topic_cols]

# # usage
X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [18]:
from itertools import product
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

def grid_search(
    X,
    y,
    param_grid,
    verbose=True,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    return_details=True,
):
    """
    Parallel grid search.

    Parameters
    ----------
    return_details : bool, default True
        If False, skip collecting and concatenating window-level details
        (MUCH faster and lower memory usage).

    Returns
    -------
    results_df : pd.DataFrame
        Summary metrics for each configuration.
    coefficients_df : pd.DataFrame or None
        Detailed coefficients if return_details=True, else None.
    """

    combos = list(product(
        param_grid["window_sizes"],
        param_grid["n_lags"],
        param_grid["lambdas"],
    ))

    if verbose:
        print(f"Testing {len(combos)} configurations...")

    # -------------------------------------------------
    # Wrapper to prevent single failure from killing run
    # -------------------------------------------------
    def _safe_run(args):
        w, L, lam = args
        try:
            return estimate_single_config(X, y, w, L, lam)
        except Exception as e:
            if verbose:
                print(f"❌ Failed config (w={w}, L={L}, λ={lam}): {e}")
            return None

    iterator = tqdm(combos, desc="Grid search") if verbose else combos

    # -------------------------------------------------
    # Parallel execution
    # -------------------------------------------------
    results = Parallel(n_jobs=n_jobs, backend=backend, prefer=prefer)(
        delayed(_safe_run)(args) for args in iterator
    )

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    summary_list = []
    details_list = [] if return_details else None

    for res in results:
        if res is None:
            continue

        summary_list.append(res.get("summary", {}))

        if return_details:
            det = res.get("details", None)
            if det is not None and not det.empty:
                details_list.append(det)

    # -------------------------------------------------
    # Build summary DataFrame
    # -------------------------------------------------
    results_df = pd.DataFrame(summary_list)

    if not results_df.empty and "r2_oos_stage2" in results_df.columns:
        results_df = results_df.sort_values(
            "r2_oos_stage2", ascending=False
        ).reset_index(drop=True)

    # -------------------------------------------------
    # Build details DataFrame (optional)
    # -------------------------------------------------
    coefficients_df = None
    if return_details and details_list:
        coefficients_df = pd.concat(details_list, ignore_index=True)

    # -------------------------------------------------
    # Reporting
    # -------------------------------------------------
    if verbose:
        print("\n" + "=" * 80)
        print("GRID SEARCH COMPLETE")
        print("=" * 80)

        if "kappa" in results_df.columns:
            n_failed = results_df["kappa"].isna().sum()
            if n_failed > 0:
                print(f"⚠️  {n_failed}/{len(results_df)} configurations failed")

    return results_df, coefficients_df


## Grid Search

In [19]:
import numpy as np
import pandas as pd

# -------------------------------------------------
# Hierarchical objective (grid-search version)
#   - If t < TSTAT_MIN: score = t - TSTAT_MIN  (negative; closer to 0 is better)
#   - If t >= TSTAT_MIN: score = max(r2, 0)    (non-negative; always beats insignificant)
#   - If hard prune and t > TSTAT_MAX: treat as invalid / worst
#   - If any NaN/inf: worst
# -------------------------------------------------
TSTAT_MIN = 1.96
TSTAT_MAX = 100.0
HARD_PRUNE_TOO_LARGE = True
WORST_SCORE = -999.0

def objective_function_hierarchical(
    row,
    r2_col="r2_insample_stage2",
    t_col="kappa_tstat",
    kappa_col="kappa",
    tstat_min=TSTAT_MIN,
    tstat_max=TSTAT_MAX,
    hard_prune=HARD_PRUNE_TOO_LARGE,
    worst=WORST_SCORE,
):
    r2 = pd.to_numeric(row.get(r2_col, np.nan), errors="coerce")
    t = pd.to_numeric(row.get(t_col, np.nan), errors="coerce")
    kappa = pd.to_numeric(row.get(kappa_col, np.nan), errors="coerce")

    # Safety checks
    if not (np.isfinite(r2) and np.isfinite(t) and np.isfinite(kappa)):
        return worst, "invalid_nan"

    # Hard prune instability
    if hard_prune and (t > tstat_max):
        return worst, "pruned_instability"

    # Case A: not significant -> guide optimizer to increase t-stat
    if t < tstat_min:
        # Example: t=0 => -1.96 ; t=1.95 => -0.01
        return float(t - tstat_min), "insignificant"

    # Case B: significant -> maximize R2 (clamp broken negatives)
    r2 = float(max(r2, 0.0))
    return r2, "significant_optimize_r2"


# -------------------------------------------------
# Iterative refinement grid search
# -------------------------------------------------
# Initial coarse grid
param_grid = {
    "window_sizes": [36, 52, 104, 208],
    "n_lags": [1, 4, 8, 12],
    "lambdas": [0.0001, 0.0005, 0.001],
}

max_iterations = 3
best_overall = None
all_results = []

for iteration in range(max_iterations):
    print(f"\n{'='*60}")
    print(f"Iteration {iteration + 1}/{max_iterations}")
    print(f"{'='*60}")

    # Run grid search (assumes your grid_search returns summary_df with cols:
    # ['window_size','n_lags','lambda','r2_insample_stage2','kappa_tstat','kappa', ...]
    summary_df, _ = grid_search(
        X, y, param_grid,
        verbose=True,
        return_details=False
    )

    # Compute hierarchical objective + status
    obj_and_status = summary_df.apply(
        lambda row: objective_function_hierarchical(row),
        axis=1,
        result_type="expand"
    )
    obj_and_status.columns = ["objective", "status"]
    summary_df = pd.concat([summary_df, obj_and_status], axis=1)

    # (Optional) keep raw values handy for debugging
    summary_df["r2_raw"] = pd.to_numeric(summary_df.get("r2_insample_stage2"), errors="coerce")
    summary_df["kappa_tstat_raw"] = pd.to_numeric(summary_df.get("kappa_tstat"), errors="coerce")

    summary_df["iteration"] = iteration + 1
    all_results.append(summary_df)

    # Select best configuration from this iteration
    valid = summary_df[summary_df["objective"] > WORST_SCORE]

    if valid.empty:
        print("No valid configurations found in this iteration.")
        break

    best_iter = valid.loc[valid["objective"].idxmax()]
    print(f"\nBest in iteration {iteration + 1}:")
    print(best_iter[[
        "window_size", "n_lags", "lambda",
        "objective", "status",
        "r2_insample_stage2", "kappa_tstat", "kappa"
    ]])

    # Update global best
    if best_overall is None or best_iter["objective"] > best_overall["objective"]:
        best_overall = best_iter

    # Stop if this is the last iteration
    if iteration == max_iterations - 1:
        break

    # -------------------------------------------------
    # Refine grid around best parameters
    # -------------------------------------------------
    best_window = int(best_iter["window_size"])
    best_lag = int(best_iter["n_lags"])
    best_lambda = float(best_iter["lambda"])

    # Finer granularity around current best
    window_step = max(4, (max(param_grid["window_sizes"]) - min(param_grid["window_sizes"])) // 4)
    lag_step = max(1, (max(param_grid["n_lags"]) - min(param_grid["n_lags"])) // 4)

    param_grid = {
        "window_sizes": sorted(set([
            max(12, best_window - window_step),
            best_window,
            best_window + window_step
        ])),
        "n_lags": sorted(set([
            max(1, best_lag - lag_step),
            best_lag,
            best_lag + lag_step
        ])),
        "lambdas": sorted(set([
            max(1e-6, best_lambda / 2),
            best_lambda,
            best_lambda * 2
        ]))
    }

    print(f"\nRefined grid for next iteration:")
    print(f"  window_sizes: {param_grid['window_sizes']}")
    print(f"  n_lags: {param_grid['n_lags']}")
    print(f"  lambdas: {param_grid['lambdas']}")

# -------------------------------------------------
# Combine all results
# -------------------------------------------------
summary_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# -------------------------------------------------
# Display final best configuration
# -------------------------------------------------
if best_overall is not None:
    print(f"\n{'='*60}")
    print("FINAL BEST CONFIGURATION")
    print(f"{'='*60}")
    print(best_overall[[
        "window_size", "n_lags", "lambda",
        "objective", "status", "iteration",
        "r2_insample_stage2", "kappa_tstat", "kappa"
    ]])
else:
    print("\nNo valid configurations found across all iterations.")

# -------------------------------------------------
# Return results
# -------------------------------------------------
summary_df, best_overall




Iteration 1/3
Testing 48 configurations...


Grid search: 100%|██████████| 48/48 [01:30<00:00,  1.88s/it]



GRID SEARCH COMPLETE

Best in iteration 1:
window_size                     208
n_lags                           12
lambda                       0.0001
objective                 -0.087597
status                insignificant
r2_insample_stage2         0.001934
kappa_tstat                1.872403
kappa                      0.041925
Name: 0, dtype: object

Refined grid for next iteration:
  window_sizes: [165, 208, 251]
  n_lags: [10, 12, 14]
  lambdas: [5e-05, 0.0001, 0.0002]

Iteration 2/3
Testing 27 configurations...


Grid search: 100%|██████████| 27/27 [00:00<00:00, 1472.89it/s]


KeyboardInterrupt: 

In [ ]:
# import numpy as np
# import pandas as pd

# # -------------------------------------------------
# # Objective function: maximize R^2 only
# # -------------------------------------------------
# def objective_function(row, r2_col="r2_insample_stage2"):
#     r2 = pd.to_numeric(row.get(r2_col, np.nan), errors="coerce")
#     if not np.isfinite(r2):
#         return -1.0
#     return r2


# # -------------------------------------------------
# # Iterative grid refinement
# # -------------------------------------------------
# results_all = []
# prev_best = -np.inf

# param_grid = {
#     "window_sizes": [36, 52, 78, 104, 150, 200],
#     "n_lags": [1, 4, 8, 12],
#     "lambdas": [0.0001, 0.00001],
# }

# for iteration in range(5):

#     # Run grid search
#     summary_df, _ = grid_search(X, y, param_grid, verbose=True)

#     # Compute objective
#     summary_df["objective"] = summary_df.apply(objective_function, axis=1)
#     results_all.append(summary_df)

#     # Check for valid candidates
#     valid_scores = summary_df.loc[summary_df["objective"] > -1.0, "objective"]
#     if valid_scores.empty:
#         break

#     best_idx = valid_scores.idxmax()
#     best_obj = valid_scores.max()

#     # Convergence check
#     if iteration > 0 and (best_obj - prev_best) < 1e-6:
#         break

#     prev_best = best_obj
#     best = summary_df.loc[best_idx]

#     # -------------------------------------------------
#     # Refine grid around best point
#     # -------------------------------------------------
#     w = int(best["window_size"])
#     l = int(best["n_lags"])
#     lam = float(best["lambda"])

#     param_grid = {
#         "window_sizes": sorted(
#             {int(w * f) for f in [0.75, 0.9, 1.0, 1.1, 1.25] if w * f >= 20}
#         ),
#         "n_lags": sorted({max(1, l - 2), l - 1, l, l + 1, l + 2}),
#         "lambdas": lam * np.array([0.5, 0.75, 1.0, 1.25, 1.5]),
#     }


# # -------------------------------------------------
# # Select best overall configuration
# # -------------------------------------------------
# final = pd.concat(results_all, ignore_index=True)
# final["objective"] = final.apply(objective_function, axis=1)

# best_overall = final.loc[final["objective"].idxmax()]
# print(best_overall[["window_size", "n_lags", "lambda", "objective"]])


## Bayesian Optimization with optuna

For each hyperparameter triplet we do the following:

    - run stage1 and stage2
    - collect r2 in-sample stage2 and kappa
    - compute objective function: r2*kappa
    - let optuna try 150 random/learned samples to maximize the objective function

In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

# -----------------------------
# Configuration
# -----------------------------
TSTAT_MIN = 1.96
TSTAT_MAX = 60.0
HARD_PRUNE_TOO_LARGE = True 

def objective(trial):
    # Search space
    window_size = trial.suggest_int("window_size", 30, 300, step=10)
    n_lags = trial.suggest_int("n_lags", 1, 15)
    lam = trial.suggest_float("lambda", 1e-5, 1e-2, log=True)

    # Run estimation
    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    r2 = float(summary.get("r2_insample_stage2", np.nan))
    t = float(summary.get("kappa_tstat", np.nan))
    kappa = float(summary.get("kappa", np.nan))

    # 1. Safety Checks (NaNs)
    if not (np.isfinite(r2) and np.isfinite(t) and np.isfinite(kappa)):
        return -999.0  # Return worst possible score instead of pruning errors silently

    # Store for analysis
    trial.set_user_attr("r2_raw", r2)
    trial.set_user_attr("kappa_tstat", t)

    # 2. Hard Prune (Instability check)
    if HARD_PRUNE_TOO_LARGE and t > TSTAT_MAX:
        trial.set_user_attr("status", "pruned_instability")
        raise optuna.TrialPruned()

    # -----------------------------------------------------------
    # 3. HIERARCHICAL SCORING (The Logic Fix)
    # -----------------------------------------------------------
    
    # CASE A: Not Significant (t < 1.96)
    # Goal: Guide the optimizer purely to increase t-stat.
    # We return a negative score so it never beats a significant trial.
    # Score range: (-inf, -0.0]
    if t < TSTAT_MIN:
        trial.set_user_attr("status", "insignificant")
        # Penalty is proportional to distance from threshold.
        # e.g., if t=0, score=-1.96. If t=1.95, score=-0.01.
        return t - TSTAT_MIN 

    # CASE B: Significant (t >= 1.96)
    # Goal: Maximize R2. 
    # We assume valid R2 is >= 0. If R2 < 0, it's a broken model, treated as 0.
    # Score range: [0.0, 1.0] (assuming R2 is 0-1)
    if r2 < 0:
        r2 = 0.0
        
    trial.set_user_attr("status", "significant_optimize_r2")
    return r2

# -----------------------------
# Setup
# -----------------------------
# No pruner needed since we don't have intermediate steps
sampler = TPESampler(seed=42, multivariate=True) 
study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(objective, n_trials=500, n_jobs=6)

c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-01-27 17:27:14,259] A new study created in memory with name: no-name-540b64f8-31e9-469f-a9e3-764f44700f38
[I 2026-01-27 17:28:10,885] Trial 0 finished with value: -1.9599596091998852 and parameters: {'window_size': 280, 'n_lags': 1, 'lambda': 2.9331994831703802e-05}. Best is trial 0 with value: -1.9599596091998852.
[I 2026-01-27 17:28:17,908] Trial 2 finished with value: -1.9599999773567753 and parameters: {'window_size': 70, 'n_lags': 3, 'lambda': 0.0008921532829499401}. Best is trial 0 with value: -1.9599596091998852.
[I 2026-01-27 17:28:17,937] Trial 3 finished with value: -1.0932941241109666 and parameters: {'window_size': 30, 'n_lags': 6, 'lambda': 0.004665088419704908}. Best is trial 3 with value: -1.0932941241109666.
[I 2026-01-27 17:28:35,163] Trial 5 fin

In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

TSTAT_MIN = 1.96
TSTAT_MAX = 60.0

# anti-degeneracy for stage 1
R2_STAGE1_MIN = 1e-5          # tune: start tiny
R2_STAGE1_WEIGHT = 0.05       # small tie-breaker, not a primary objective

# strong separation between feasible/infeasible
INFEASIBLE_BASE = -1000.0
FEASIBLE_BASE = 1.0

def _finite(*xs):
    return all(np.isfinite(x) for x in xs)

def objective(trial):
    window_size = trial.suggest_int("window_size", 20, 300, step=10)
    n_lags      = trial.suggest_int("n_lags", 1, 15)
    lam         = trial.suggest_float("lambda", 1e-5, 1e-2, log=True)

    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    r2_2 = float(summary.get("r2_insample_stage2", np.nan))
    t    = float(summary.get("kappa_tstat", np.nan))
    kappa= float(summary.get("kappa", np.nan))
    r2_1 = float(summary.get("r2_insample_stage1", np.nan))  # <-- add this

    # Guard
    if not _finite(r2_2, t, kappa, r2_1):
        trial.set_user_attr("status", "invalid_nan")
        return INFEASIBLE_BASE

    trial.set_user_attr("r2_stage2", r2_2)
    trial.set_user_attr("r2_stage1", r2_1)
    trial.set_user_attr("kappa_tstat", t)
    trial.set_user_attr("kappa", kappa)

    # -------------------------
    # Feasibility / constraints
    # -------------------------
    v_t = max(0.0, TSTAT_MIN - t) + max(0.0, t - TSTAT_MAX)
    v_s1 = max(0.0, R2_STAGE1_MIN - r2_1)

    violation = v_t + 10.0 * v_s1  # stage1 violation scaled to matter

    if violation > 0:
        trial.set_user_attr("status", "infeasible")
        return INFEASIBLE_BASE - violation

    # -------------------------
    # Feasible: optimize stage 2
    # -------------------------
    trial.set_user_attr("status", "feasible")

    r2_2 = max(0.0, r2_2)
    r2_1 = max(0.0, r2_1)

    # Rescale tiny R² so TPE has resolution
    score_stage2 = np.log1p(1e4 * r2_2)

    # Small tie-breaker: encourage non-degenerate learning in stage 1
    score_stage1 = np.log1p(1e4 * r2_1)

    # Tiny bonus for stronger significance (won't dominate)
    t_bonus = 1e-3 * min(t, TSTAT_MAX)

    return FEASIBLE_BASE + score_stage2 + R2_STAGE1_WEIGHT * score_stage1 + t_bonus


sampler = TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=50,
    n_ei_candidates=64
)

study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=500, n_jobs=6, gc_after_trial=True)


c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-01-27 18:49:18,810] A new study created in memory with name: no-name-aff8b45d-e8b8-4ce4-be60-dd8e112194cb
[I 2026-01-27 18:49:36,113] Trial 2 finished with value: -1001.958715883967 and parameters: {'window_size': 30, 'n_lags': 1, 'lambda': 0.001698692702620091}. Best is trial 2 with value: -1001.958715883967.
[I 2026-01-27 18:49:45,664] Trial 3 finished with value: -1001.9599994654786 and parameters: {'window_size': 230, 'n_lags': 1, 'lambda': 0.0009885571556753037}. Best is trial 2 with value: -1001.958715883967.
[I 2026-01-27 18:50:01,112] Trial 5 finished with value: -1001.9599991285182 and parameters: {'window_size': 20, 'n_lags': 4, 'lambda': 0.000942898978432539}. Best is trial 2 with value: -1001.958715883967.
[I 2026-01-27 18:50:05,722] Trial 0 finished 

In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

TSTAT_MIN = 1.96
TSTAT_MAX = 60.0

# anti-degeneracy for stage 1
R2_STAGE1_MIN = 1e-5          # tune: start tiny
R2_STAGE1_WEIGHT = 0.05       # small tie-breaker, not a primary objective

# strong separation between feasible/infeasible
INFEASIBLE_BASE = -1000.0
FEASIBLE_BASE = 1.0

def _finite(*xs):
    return all(np.isfinite(x) for x in xs)

def objective(trial):
    window_size = trial.suggest_int("window_size", 20, 300, step=10)
    n_lags      = trial.suggest_int("n_lags", 10, 25)
    lam         = trial.suggest_float("lambda", 1e-5, 1e-2, log=True)

    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    r2_2 = float(summary.get("r2_insample_stage2", np.nan))
    t    = float(summary.get("kappa_tstat", np.nan))
    kappa= float(summary.get("kappa", np.nan))
    r2_1 = float(summary.get("r2_insample_stage1", np.nan))  # <-- add this

    # Guard
    if not _finite(r2_2, t, kappa, r2_1):
        trial.set_user_attr("status", "invalid_nan")
        return INFEASIBLE_BASE

    trial.set_user_attr("r2_stage2", r2_2)
    trial.set_user_attr("r2_stage1", r2_1)
    trial.set_user_attr("kappa_tstat", t)
    trial.set_user_attr("kappa", kappa)

    # -------------------------
    # Feasibility / constraints
    # -------------------------
    v_t = max(0.0, TSTAT_MIN - t) + max(0.0, t - TSTAT_MAX)
    v_s1 = max(0.0, R2_STAGE1_MIN - r2_1)

    violation = v_t + 10.0 * v_s1  # stage1 violation scaled to matter

    if violation > 0:
        trial.set_user_attr("status", "infeasible")
        return INFEASIBLE_BASE - violation

    # -------------------------
    # Feasible: optimize stage 2
    # -------------------------
    trial.set_user_attr("status", "feasible")

    r2_2 = max(0.0, r2_2)
    r2_1 = max(0.0, r2_1)

    # Rescale tiny R² so TPE has resolution
    score_stage2 = np.log1p(1e4 * r2_2)

    # Small tie-breaker: encourage non-degenerate learning in stage 1
    score_stage1 = np.log1p(1e4 * r2_1)

    # Tiny bonus for stronger significance (won't dominate)
    t_bonus = 1e-3 * min(t, TSTAT_MAX)

    return FEASIBLE_BASE + score_stage2 + R2_STAGE1_WEIGHT * score_stage1 + t_bonus


sampler = TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=50,
    n_ei_candidates=64
)

study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=500, n_jobs=6, gc_after_trial=True)
